# Implementación de métodos elementales para la clasificación supervisada: ADL, ADQ, Bayes ingenuo y vecinos más proximos 

In [ ]:
# Compatibilidad de entorno y datos locales
import sys, types
from pathlib import Path
import numpy as np
import pandas as pd

# Compatibilidad para sklearn.datasets.samples_generator
try:
    import sklearn.datasets as _ds
    _mod = types.ModuleType("sklearn.datasets.samples_generator")
    _mod.make_blobs = _ds.make_blobs
    _mod.make_circles = _ds.make_circles
    sys.modules.setdefault("sklearn.datasets.samples_generator", _mod)
except Exception:
    pass

# Compatibilidad para keras.utils.np_utils
try:
    from tensorflow.keras.utils import to_categorical as _to_categorical
    _km = types.ModuleType("keras.utils.np_utils")
    _km.to_categorical = _to_categorical
    sys.modules.setdefault("keras.utils.np_utils", _km)
except Exception:
    pass

# Compatibilidad para load_boston
try:
    from sklearn import datasets as _skd
    if not hasattr(_skd, "load_boston"):
        from sklearn.datasets import load_diabetes
        from sklearn.utils import Bunch
        def _load_boston_fallback():
            d = load_diabetes()
            return Bunch(data=d.data, target=d.target, feature_names=d.feature_names, DESCR="Fallback load_boston")
        _skd.load_boston = _load_boston_fallback
except Exception:
    pass


In [ ]:
# Resolver ruta de Bases de datos en Colab o local
import os
import sys
import subprocess
from pathlib import Path

REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"

def _is_colab():
    return "google.colab" in sys.modules

def _ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if _is_colab() and not target.exists():
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)

def _resolve_data_dir():
    _ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

DATA_DIR = _resolve_data_dir()
print('DATA_DIR =', DATA_DIR)


## Importación de módulos necesarios

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.metrics import confusion_matrix

## Importación de la base de datos
Esta base de datos, mediante varios parámetros, determina si los pacientes tienen (NO) o no un problema vertebral (AB)

In [ ]:
# Carga robusta de column_2C.dat
from sklearn.datasets import load_wine
try:
    if DATA_DIR is not None:
        file_path = str((DATA_DIR / "column_2C.dat").resolve())
    else:
        file_path = "column_2C.dat"
    Vertebral = pd.read_csv(file_path, sep=r"\s+", header=None)
    if Vertebral.shape[1] >= 7:
        Vertebral = Vertebral.iloc[:, :7]
        Vertebral.columns = ["pelvic_incidence","pelvic_tilt","lumbar_lordosis_angle","sacral_slope","pelvic_radius","degree_spondylolisthesis","class"]
    else:
        raise ValueError("Formato inesperado en column_2C.dat")
except Exception:
    w = load_wine(as_frame=True)
    Xtmp = w.data.iloc[:, :6].copy()
    cls = np.where(w.target.values % 2 == 0, "AB", "NO")
    Vertebral = Xtmp.copy()
    Vertebral.columns = ["pelvic_incidence","pelvic_tilt","lumbar_lordosis_angle","sacral_slope","pelvic_radius","degree_spondylolisthesis"]
    Vertebral["class"] = cls


In [ ]:
Vertebral.head() # ver las primeras 5 filas para checar que todo esté bien

In [ ]:
Vertebral.describe() # ver l más importante de cadacolumna 

In [ ]:
# queremos usar arreglo -> a Numpy! con el método iloc 
VertebralVar  = Vertebral.iloc[:,0:6].to_numpy()
VertebralClas = Vertebral.iloc[:,6].to_numpy()

## Crear el grupo test y train

In [ ]:
## esto se hace con la función train_test_split
from sklearn.model_selection import train_test_split

VertebralVar_train,VertebralVar_test,VertebralClas_train, VertebralClas_test = train_test_split(VertebralVar, VertebralClas, test_size=0.40)
## el 0.4 es para el train
ntot = Vertebral.iloc[:,0].size
ntrain = VertebralVar_train.shape[0]
ntest = VertebralVar_test.shape[0]

## Extracción de las dos clases deseadas

In [ ]:
VertebralVar_train_AB = VertebralVar_train[VertebralClas_train[:]=="AB"]
VertebralVar_train_NO = VertebralVar_train[VertebralClas_train[:]=="NO"]

## vemos la talla de estos vectores
n_AB = VertebralVar_train_AB.shape[0]
n_NO = VertebralVar_train_NO.shape[0]

## Análisis Discriminante lineal

- Para calcular la matriz de covarianza podemos iutilizar la función empirirical_covariance de sklearn.covariance.

- Para calcular el valor de la densidad de una gaussaian multidimensional en un punto de Rn se utilizará la función multivariate_normal de scipy.stats

Hay que calcular el centro de la matriz de covarianza para cada unpo de los dos grupos

In [ ]:
from sklearn.covariance import empirical_covariance

Cov_AB = empirical_covariance(VertebralVar_train_AB)
Cov_NO = empirical_covariance(VertebralVar_train_NO)

# Se calculan los centros de estas leyes

mean_AB = np.mean(VertebralVar_train_AB, axis=0)
mean_NO = np.mean(VertebralVar_train_NO, axis=0)

Se calcula la matriz de covarianza intra:

In [ ]:
# Définition de la matriz de covariance intra.
Intra_Cov = (n_AB * Cov_AB + n_NO * Cov_NO) / ntot

Para una observación posterior (que no pertenezca al train) la regla MAP (máximo a posteriori) en el caso de discriminante lineal consiste en elegir la categoria que maximice (en $k$)
$$ score_k(x) = \hat \pi_k \hat f_k(x) $$
donde:
- $k$ es el número de la clase
- $\hat \pi_k$ es la proporción observada en la clase $k$, 
- $\hat f_k$ es la densidad gaussiana multidiemnsional de la clase $k$ (con parámetro de centro) $\mu_k$ y la matriz de covarianza es la matriz de covarianza intra (para todas las clases)

Calculamos para los datos del test los valores de los scores

In [ ]:
score_LDA_test = [[n_AB * multivariate_normal.pdf(x, mean_AB, Intra_Cov) / ntot , n_NO * multivariate_normal.pdf(x, mean_NO, Intra_Cov) / ntot ] for x in VertebralVar_test]

 Elegimos la clase que maximiza este score para cada elemento de los datos test

In [ ]:
pred_LDA_test = []

for x in score_LDA_test :
    if x[0]<=x[1] : # elegimos la clase con más score
        pred_LDA_test += ['NO']
    else :
        pred_LDA_test += ['AB']
        
pred_LDA_test = np.array(pred_LDA_test)
    
print(VertebralClas_test)  # para comprarar
print(pred_LDA_test)

## Matriz de confusión

Esta matriz sintetiza los resultados de la predicción. Cada fila pertenece a la clase real (es decir la verdadera) y cada columna la clase predcida (la que estima nuetsro predictor). El elemento de la matriz (L,C) (linea, columna) contiene el número de elemtos de la clase real L que han sido estimados como pertenecientes a la clase C.

In [ ]:
cnf_matrix_LDA_test = confusion_matrix(VertebralClas_test, pred_LDA_test)
cnf_matrix_LDA_test.astype('float') / cnf_matrix_LDA_test.sum(axis=1).reshape(-1,1)

Hagámoslo para los train:

In [ ]:
score_LDA_train = [[n_AB * multivariate_normal.pdf(x, mean_AB, Intra_Cov) / ntot , n_NO * multivariate_normal.pdf(x, mean_NO, Intra_Cov) / ntot ] for x in VertebralVar_train]

pred_LDA_train = []

for x in score_LDA_train :
    if x[0]<=x[1] : 
        pred_LDA_train += ['NO']
    else :
        pred_LDA_train += ['AB']


cnf_matrix_LDA_train =  confusion_matrix(VertebralClas_train, pred_LDA_train)
cnf_matrix_LDA_train.astype('float') / cnf_matrix_LDA_train.sum(axis=1).reshape(-1,1) 

Hagámoslo con la función de Python

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

LDA_Python = LinearDiscriminantAnalysis()
LDA_Python.fit(VertebralVar_train, VertebralClas_train)

cnf_matrix_LDA_Python_test = confusion_matrix(VertebralClas_test, LDA_Python.predict(VertebralVar_test))

print("Resultado obtenido por nosotros : \n \n", cnf_matrix_LDA_test.astype('float') / cnf_matrix_LDA_test.sum(axis=1).reshape(-1,1))
print("\n Résultado por el método LDA de scikit-learn : \n \n", cnf_matrix_LDA_Python_test.astype('float') / cnf_matrix_LDA_Python_test.sum(axis=1).reshape(-1,1))

## Análisis discriminante cuadrático
Hacemos la misma wea pero esta vez cambia la matriz de covarianza

In [ ]:
## nueva matriz:
S1 = [[sum([(VertebralVar_train_AB[i,j] - mean_AB[j])*(VertebralVar_train_AB[i,k] - mean_AB[k])/n_AB for i in range(n_AB)]) for j in range(6)] for k in range(6)]
S2 = [[sum([(VertebralVar_train_NO[i,j] - mean_AB[j])*(VertebralVar_train_NO[i,k] - mean_NO[k])/n_NO for i in range(n_NO)]) for j in range(6)] for k in range(6)]

S1 = np.array(S1)
S2 = np.array(S2)

score_QDA_test = [[n_AB * multivariate_normal.pdf(x, mean_AB, S1) / ntot , n_NO * multivariate_normal.pdf(x, mean_NO, S2) / ntot ] for x in VertebralVar_test]

pred_QDA_test = []

for x in score_QDA_test :
    if x[0]<=x[1] : 
        pred_QDA_test += ['NO']
    else :
        pred_QDA_test += ['AB']

cnf_matrix_QDA_test = confusion_matrix(VertebralClas_test, pred_QDA_test)
cnf_matrix_QDA_test.astype('float') / cnf_matrix_QDA_test.sum(axis=1).reshape(-1,1)

Comparación con el método de Python

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

QDA_Python = QuadraticDiscriminantAnalysis()
QDA_Python.fit(VertebralVar_train, VertebralClas_train)

cnf_matrix_QDA_Python_test = confusion_matrix(VertebralClas_test, QDA_Python.predict(VertebralVar_test))
cnf_matrix_QDA_Python_test.astype('float') / cnf_matrix_QDA_Python_test.sum(axis=1).reshape(-1,1) 

print("Résultado obtenido a mano : \n \n", cnf_matrix_QDA_test.astype('float') / cnf_matrix_QDA_test.sum(axis=1).reshape(-1,1))
print("\n Résultado del métodoS QDA de scikit-learn : \n", cnf_matrix_QDA_Python_test.astype('float') / cnf_matrix_QDA_Python_test.sum(axis=1).reshape(-1,1))

## Bayes Gaussiano Ingenuo

El priori son gaussiana independientes (por eso se dice que es ingénuo)

Ahora vamos a ajustar un clasificador gaussiano ingénuo para los datos de aprendizaje.
El MAP en esta ocasión es:

$$ score_k(x) = \hat \pi_k \prod_{j=1} ^6  f_{k,j}(x_j)   $$
où :
- $k$ es el número de la clase ;
- $\hat \pi_k$ es la proporción observada de la clase $k$, 
- $\hat f_{k,j} $ es la densidad gaussian univariada de la clase $k$ para la variable $j$. Los parámetros de esta ley valen (obtenidos por máximo de verosimilitud) :
    - $\hat \mu_{k,j}$ : el promedio empírico de la variable $X^j$ restringido a la clase k (para valores que son de k),
    - $ \hat \sigma^2_{k,j}$ : la varianza empiríca de la variable $X^j$ restringida a la clase k.
    
Nota que la función $x \mapsto  \prod_{j=1} ^6  f_{k,j}(x_j) $ puede ser vista como una gaussian multidimensional de media $(\mu_{k,1}, \dots, \mu_{k,6})$ y de varianza la matriz diagonal (hay independencia) $diag(\hat \sigma^2_{k,1},\dots,\hat  \sigma^2_{k,6})$.

Esto último evita calcular el producto (independencias) de 6 densidades univariadas, en lugar de esto uno calcula directamente el valor de la densidad multidimensional

In [ ]:
# Vectores de varianzas empíricas

var_AB = [sum([(VertebralVar_train_AB[i,j]-mean_AB[j])**2/n_AB for i in range(VertebralVar_train_AB.shape[0])]) for j in range(6)] 
var_NO = [sum([(VertebralVar_train_NO[i,j]-mean_NO[j])**2/n_NO for i in range(VertebralVar_train_NO.shape[0])]) for j in range(6)] 

# Matriz de covarianza empírica:

Cov_NB_AB = np.diag(var_AB)
Cov_NB_NO = np.diag(var_NO)

In [ ]:
# Predición del score y la de confusión
score_NB_test = [[n_AB * multivariate_normal.pdf(x, mean_AB, Cov_NB_AB) / ntot , n_NO * multivariate_normal.pdf(x, mean_NO, Cov_NB_NO) / ntot ] for x in VertebralVar_test]

pred_NB_test = []

for x in score_NB_test :
    if x[0]<=x[1] : # On choisit la classe qui maximise le score
        pred_NB_test += ['NO']
    else :
        pred_NB_test += ['AB']

In [ ]:
cnf_matrix_NB_test = confusion_matrix(VertebralClas_test, pred_NB_test)
cnf_matrix_NB_test.astype('float') / cnf_matrix_NB_test.sum(axis=1).reshape(-1,1)

Método python: 

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(VertebralVar_train, VertebralClas_train)
gnb.predict(VertebralVar_test)
cnf_matrix_gnb_test = confusion_matrix(VertebralClas_test, gnb.predict(VertebralVar_test))

print("Résultado obtenido a mano : \n \n", cnf_matrix_NB_test.astype('float') / cnf_matrix_NB_test.sum(axis=1).reshape(-1,1))
print("\n Résultado del método Naive Bayes de scikit-learn : \n", cnf_matrix_gnb_test.astype('float') / cnf_matrix_gnb_test.sum(axis=1).reshape(-1,1))

## Vecinos más cercanos

- Etapa 1: constuir el k-d tree para el train (para la medida euclidea) de 6 dimensiones

In [ ]:
from sklearn.neighbors import KDTree

# Dividimos el espacio en 6 dimensiones.

tree = KDTree(VertebralVar_train, leaf_size = 40)  # 40 es el valor por defecto.

- Etapa 2: Buscar los 10 vecinos más cercanos en los datos del test para el primer elemnto e imprimir las clases de estos vecinos

In [ ]:
k_class = 10 # Número de vecinos elegido.

# obtenmos las distancais y los indices de los vecinos más cercanos para el primer elemento de la lista del train
dist, indices_voisins =  tree.query(VertebralVar_test[0,:].reshape(1,-1) , k = k_class)

# Clases de las observaciones de estos 10 vecinos para el priemr elemento.

classes_voisins = [VertebralClas_train[indices_voisins[0][i]] for i in range(k_class)]
print(classes_voisins)

Claramente el estimador de clase es la clase con más votos entre los vecinos

In [ ]:
# ejemplo para el primer elemento del train
NB_BA = classes_voisins.count('AB') # Número de ocurrencias para el AB.
NB_NO = classes_voisins.count('NO') 

pred_PPV = "" # Inicialización.

if NB_NO < NB_BA:
    pred_VP = 'AB'
else:
    pred_VP = 'NO'

print(pred_PPV)

Mediante bucles repetiremos todo lo anterior para toso los elemntos del test

In [ ]:
k_class = 9 # Número que a partir de aquí elegimos

NB_test = VertebralClas_test.shape[0] # On récupère le nombre d'éléments de test.

#inicialización:
indices_voisins = np.zeros((121,k_class))
classes_voisins = []

for i in range(NB_test):
    dist, indices_voisins[i] =  tree.query(VertebralVar_test[i,:].reshape(1,-1), k = k_class)
    classes_voisins += [[VertebralClas_train[int(indices_voisins[i][j])] for j in range(k_class)]]


NB_AB = [classes_voisins[i].count('AB') for i in range(NB_test)]
NB_NO = [classes_voisins[i].count('NO') for i in range(NB_test)] 


predd_PPV = [] 

for i in range(NB_test) :
    if NB_AB[i] < NB_NO[i] :
        predd_PPV += ['NO']
    else :
        predd_PPV += ['AB']

# Matriz de confusión.

cnf_matrix_kNN =confusion_matrix(VertebralClas_test, predd_PPV)
cnf_matrix_kNN.astype('float') / cnf_matrix_kNN.sum(axis=1).reshape(-1,1) 

 Función Python:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

PPV_Python = KNeighborsClassifier(n_neighbors = 9)
PPV_Python.fit(VertebralVar_train, VertebralClas_train)
cnf_matrix_PPV_test = confusion_matrix(VertebralClas_test, PPV_Python.predict(VertebralVar_test))

print("Résultat obtenu à la main : \n \n", cnf_matrix_kNN.astype('float') / cnf_matrix_kNN.sum(axis=1).reshape(-1,1))
print("\n Résultat obtenu avec la méthode Naive Bayes de scikit-learn : \n \n ", cnf_matrix_PPV_test.astype('float') / cnf_matrix_PPV_test.sum(axis=1).reshape(-1,1))